# MIRA: Euclidean vs symRMSD metric comparison

This notebook compares the **existing** MIRA scores (computed with flat Euclidean /
unaligned-RMSD distance in normalised coordinate space) against the **new**
`symrmsd` metric (symmetry-corrected heavy-atom RMSD via spyrmsd).

Goals:
1. Confirm there is no implementation bug (scores should be correlated, not
   identical — they measure the same calibration concept with different distance
   metrics).
2. Quantify the effect of symmetry correction (symmetric ligands should show
   the largest difference between old and new scores).

**Dataset:** DiffDock PoseBusters benchmark (308 complexes, 40 samples each).

**Runtime note:** The pairwise symRMSD matrix for each complex involves up to
S×(S+1)/2 = 820 spyrmsd calls (S=40). For a small subset of ~20 complexes this
takes ~1-3 min. For the full 308-complex set use the SLURM script.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

DIFFDOCK = Path('/home/qf226/MProject/DiffDock')
sys.path.insert(0, str(DIFFDOCK))

from utils.tarp_eval import build_results_index
from utils.mira_eval import compute_mira_scores, mira_null

# ── paths ──────────────────────────────────────────────────────────────────────
RESULTS_DIR  = Path('/home/qf226/rds/hpc-work/results/DiffDock/pb_evaluate_v2')
METRICS_DIR  = Path('/home/qf226/rds/hpc-work/results/DiffDock/pb_evaluate_v2_merged/metrics')
DATA_DIR     = '/home/qf226/rds/hpc-work/data/posebusters_benchmark_set'

print('Paths set up.')

## 1. Load existing (rmsd-metric) MIRA scores

In [ ]:
old_names  = np.load(METRICS_DIR / 'mira_names_rmsd.npy',   allow_pickle=True)
old_scores = np.load(METRICS_DIR / 'mira_scores_rmsd.npy',  allow_pickle=True).astype(float)

print(f'Loaded {len(old_names)} complexes with existing rmsd-metric MIRA scores.')
print(f'  Mean score = {old_scores.mean():.4f}   (null ≈ {mira_null(40):.4f})')

## 2. Compute new symRMSD MIRA scores on a small subset

Run on the first 20 complexes shared between the old results and the results index.
Uses `metric='symrmsd'` which bypasses `mira_score.mira` and calls spyrmsd directly.

In [ ]:
N_SUBSET = 20  # increase for a more thorough check; full 308 → use SLURM

results_index = build_results_index(str(RESULTS_DIR))
print(f'Results index: {len(results_index)} complexes.')

# Pick complexes that exist in both old scores AND the results index
available = [n for n in old_names if n in results_index]
subset    = available[:N_SUBSET]
print(f'Running symRMSD MIRA on {len(subset)} complexes ...')

In [ ]:
new_names, new_scores = compute_mira_scores(
    subset, results_index, DATA_DIR,
    num_runs=100, metric='symrmsd', seed=42, verbose=True
)
print(f'\nNew symRMSD MIRA: {len(new_scores)} complexes evaluated.')
print(f'  Mean score = {new_scores.mean():.4f}   (null ≈ {mira_null(40):.4f})')

## 3. Match scores for the shared subset

In [ ]:
# Align by complex name
old_map = dict(zip(old_names, old_scores))
new_map = dict(zip(new_names, new_scores))
common  = sorted(set(old_map) & set(new_map))

scores_old = np.array([old_map[n] for n in common])
scores_new = np.array([new_map[n] for n in common])

print(f'Matched {len(common)} complexes.')
print(f'  rmsd-metric:  mean = {scores_old.mean():.4f}')
print(f'  symrmsd:      mean = {scores_new.mean():.4f}')
print(f'  null (S=40) = {mira_null(40):.4f}')
print(f'  Mean abs diff = {np.abs(scores_old - scores_new).mean():.4f}')

## 4. Comparison plots

In [ ]:
null = mira_null(40)

fig = plt.figure(figsize=(13, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

# ── scatter ──────────────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
ax0.scatter(scores_old, scores_new, s=40, alpha=0.7, edgecolors='none')
lo, hi = min(scores_old.min(), scores_new.min()) - 0.02, \
         max(scores_old.max(), scores_new.max()) + 0.02
ax0.plot([lo, hi], [lo, hi], 'k--', lw=1, label='identity')
ax0.axvline(null, color='C1', lw=1, ls=':', label=f'null={null:.3f}')
ax0.axhline(null, color='C1', lw=1, ls=':')
ax0.set_xlabel('MIRA (rmsd metric)')
ax0.set_ylabel('MIRA (symrmsd metric)')
ax0.set_title('Per-complex scores')
ax0.legend(fontsize=8)

# ── histograms ────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
bins = np.linspace(min(scores_old.min(), scores_new.min()) - 0.02,
                   max(scores_old.max(), scores_new.max()) + 0.02, 25)
ax1.hist(scores_old, bins=bins, alpha=0.6, label='rmsd',    color='C0')
ax1.hist(scores_new, bins=bins, alpha=0.6, label='symrmsd', color='C2')
ax1.axvline(null, color='k', lw=1.5, ls='--', label=f'null={null:.3f}')
ax1.axvline(scores_old.mean(), color='C0', lw=1.5, ls='-')
ax1.axvline(scores_new.mean(), color='C2', lw=1.5, ls='-')
ax1.set_xlabel('MIRA score')
ax1.set_ylabel('Count')
ax1.set_title('Score distributions')
ax1.legend(fontsize=8)

# ── per-complex difference ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
diff = scores_new - scores_old
ax2.hist(diff, bins=20, color='C3', alpha=0.7)
ax2.axvline(0, color='k', lw=1.5, ls='--', label='no change')
ax2.axvline(diff.mean(), color='C3', lw=1.5, ls='-',
            label=f'mean={diff.mean():.3f}')
ax2.set_xlabel('symrmsd − rmsd')
ax2.set_ylabel('Count')
ax2.set_title('Per-complex difference')
ax2.legend(fontsize=8)

fig.suptitle(f'MIRA metric comparison — DiffDock PB subset (N={len(common)})',
             fontsize=11)
plt.savefig(DIFFDOCK / 'notebooks' / 'mira_metric_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to notebooks/mira_metric_comparison.png')

## 5. Sanity checks

In [ ]:
# ── 5a. Scores should be finite and in a plausible range ─────────────────────
print('=== Sanity checks ===')
print(f'All symrmsd scores finite: {np.all(np.isfinite(scores_new))}')
print(f'symrmsd scores in [0.3, 1.1]: '
      f'{np.all((scores_new >= 0.3) & (scores_new <= 1.1))}')
print(f'Pearson r(old, new): {np.corrcoef(scores_old, scores_new)[0,1]:.3f}')
print()

# ── 5b. Identify complexes with the largest difference ───────────────────────
idx_sorted = np.argsort(np.abs(diff))[::-1]
print('Top-5 complexes by |symrmsd − rmsd|:')
for k in idx_sorted[:5]:
    print(f'  {common[k]:20s}  rmsd={scores_old[k]:.4f}  symrmsd={scores_new[k]:.4f}'
          f'  Δ={diff[k]:+.4f}')

In [ ]:
# ── 5c. Spot-check one complex — inspect its symRMSD matrix ──────────────────
from rdkit import Chem
from utils.tarp_eval import load_crystal_coords, load_sample_coords, compute_rmsd_symmetry

# Pick the complex with the largest absolute difference
demo_id = common[idx_sorted[0]]
print(f'Spot-checking: {demo_id}')

crystal_mol, all_crystal = load_crystal_coords(demo_id, DATA_DIR)
crystal_coords = all_crystal[0]
sample_coords  = load_sample_coords(demo_id, results_index)
S = len(sample_coords)
print(f'  {S} predicted poses, {crystal_mol.GetNumAtoms()} heavy atoms')

# Small pairwise matrix (crystal vs first 5 samples)
top5 = sample_coords[:5]
rmsds_crystal_to_samples = compute_rmsd_symmetry(crystal_mol, crystal_coords, top5)
print(f'  symRMSD(crystal, sample_1..5): {np.round(rmsds_crystal_to_samples, 3)}')

## Summary

If the implementation is correct:
- Both metrics should produce MIRA scores that are **positively correlated** (r > 0.3)
- The symRMSD mean should be close to the rmsd mean (within ~0.05)
- Complexes with the largest difference typically have **symmetric ligands** or
  ligands where atom-ordering artefacts in the SDF affect unaligned RMSD
- No scores should be NaN (for complexes that had valid scores with the old metric)